In [1]:
import numpy as np
import os
from pathlib import Path

def split_loso_cv_patient(npz_path, output_root, neg_to_pos_ratio=1, random_state=42):
    # 1. Load dữ liệu gốc
    if not os.path.exists(npz_path):
        print(f"LỖI: Không tìm thấy file tại {npz_path}")
        return

    data = np.load(npz_path)
    X, y, sids = data["X"], data["y"], data["seizure_ids"]
    sfreq, starts, ends = data["sfreq"], data["starts"], data["ends"]
    
    rng = np.random.default_rng(random_state)
    
    # 2. Xác định danh sách các cơn (Seizure IDs >= 0)
    unique_seizures = np.sort(np.unique(sids[sids >= 0]))
    n_seizures = len(unique_seizures)
    
    if n_seizures < 2:
        print("LỖI: Cần ít nhất 2 cơn để chia Train/Test.")
        return

    print(f"--- Bắt đầu LOSO-CV với {n_seizures} cơn co giật ---")

    # Pool toàn bộ dữ liệu nền (Interictal) của Patient
    idx_interictal_all = np.where(y == 0)[0]
    rng.shuffle(idx_interictal_all)

    # 3. Vòng lặp LOSO
    for i, test_sid in enumerate(unique_seizures):
        val_sid = unique_seizures[(i + 1) % n_seizures]
        train_sids = [s for s in unique_seizures if s != test_sid and s != val_sid]

        fold_dir = os.path.join(output_root, f"fold_test_sid_{test_sid}")
        os.makedirs(fold_dir, exist_ok=True)

        # Tách index Preictal (y=1) theo từng cơn
        idx_train_pos = np.where(np.isin(sids, train_sids) & (y == 1))[0]
        idx_val_pos   = np.where((sids == val_sid) & (y == 1))[0]
        idx_test_pos  = np.where((sids == test_sid) & (y == 1))[0]

        # CHIA POOL INTERICTAL ĐỘNG:
        # Lấy nhãn 0 cho Test và Val trước để đảm bảo chúng luôn có dữ liệu đối chứng
        n_test_neg = int(len(idx_test_pos) * neg_to_pos_ratio)
        n_val_neg  = int(len(idx_val_pos) * neg_to_pos_ratio)
        
        idx_test_neg = idx_interictal_all[:n_test_neg]
        idx_val_neg  = idx_interictal_all[n_test_neg : n_test_neg + n_val_neg]
        idx_train_neg_pool = idx_interictal_all[n_test_neg + n_val_neg:]

        # Hàm tạo tập dữ liệu cuối cùng
        def get_final_indices(pos_idx, neg_idx):
            final = np.concatenate([pos_idx, neg_idx]).astype(int)
            rng.shuffle(final)
            return final

        # Giới hạn nhãn 0 cho Train để cân bằng tỷ lệ
        n_train_neg = min(len(idx_train_neg_pool), int(len(idx_train_pos) * neg_to_pos_ratio))
        idx_train_neg = idx_train_neg_pool[:n_train_neg]

        train_idx = get_final_indices(idx_train_pos, idx_train_neg)
        val_idx   = get_final_indices(idx_val_pos, idx_val_neg)
        test_idx  = get_final_indices(idx_test_pos, idx_test_neg)

        # 4. Lưu và Kiểm tra phân phối nhãn
        split_data = {"train": train_idx, "validation": val_idx, "test": test_idx}

        print(f"\n--- Fold {i+1}/{n_seizures} (Test Sid: {test_sid}) ---")
        
        for name, indices in split_data.items():
            if len(indices) == 0: continue
            
            curr_y = y[indices]
            # Dòng print kiểm tra theo yêu cầu của bạn
            print(f"{name.capitalize()} Set: Preictal={sum(curr_y==1)}, Interictal={sum(curr_y==0)}")
            
            np.savez_compressed(
                os.path.join(fold_dir, f"{name}.npz"),
                X=X[indices], y=y[indices], seizure_ids=sids[indices],
                sfreq=sfreq, starts=starts[indices], ends=ends[indices]
            )

if __name__ == "__main__":
    FILE_NPZ = r"C:\Users\Admin\Documents\Thesis\Patient-Specific\chb13\chb13.npz"
    OUT_DIR  = r"C:\Users\Admin\Documents\Thesis\LOSO_CV\chb13"
    split_loso_cv_patient(FILE_NPZ, OUT_DIR)

--- Bắt đầu LOSO-CV với 6 cơn co giật ---

--- Fold 1/6 (Test Sid: 0) ---
Train Set: Preictal=1150, Interictal=1150
Validation Set: Preictal=446, Interictal=446
Test Set: Preictal=448, Interictal=448

--- Fold 2/6 (Test Sid: 2) ---
Train Set: Preictal=1152, Interictal=1152
Validation Set: Preictal=446, Interictal=446
Test Set: Preictal=446, Interictal=446

--- Fold 3/6 (Test Sid: 4) ---
Train Set: Preictal=1150, Interictal=1150
Validation Set: Preictal=448, Interictal=448
Test Set: Preictal=446, Interictal=446

--- Fold 4/6 (Test Sid: 6) ---
Train Set: Preictal=1386, Interictal=1386
Validation Set: Preictal=210, Interictal=210
Test Set: Preictal=448, Interictal=448

--- Fold 5/6 (Test Sid: 7) ---
Train Set: Preictal=1788, Interictal=1788
Validation Set: Preictal=46, Interictal=46
Test Set: Preictal=210, Interictal=210

--- Fold 6/6 (Test Sid: 9) ---
Train Set: Preictal=1550, Interictal=1550


KeyboardInterrupt: 